# Sampling origins across different cities and creating origin-destination pairs.

## Prerequisites
This repository builds on the python package OSMNx (v.2.0.1, https://osmnx.readthedocs.io/en/stable/). I recommend installing it via conda:
```
conda create -n ox -c conda-forge --strict-channel-priority osmnx
```
For sampling nodes based on city names two additional packages are required, namely geopy (v.2.3.1, https://geopy.readthedocs.io/en/stable/) and overpy (v.0.7, https://python-overpy.readthedocs.io/en/latest/)

```
pip install geopy
pip install overpy nodes run Ubuntu Jammy 22.04 LTS.
There is local scratch space on each node, which is shared between the jobs currently running. Connected to Kebnekaise is also our parallel file system Ransarn (where your project storage is located), which provide quick access to files regardless of which node they run on. For more information about the different file systems that are available on our systems, read the Filesystems and Storage page.
```

For visualizing routes and geometry on maps I use the folium package (v.0.19.4, https://python-visualization.github.io/folium/latest/) that is included in the OSMNx package, but for creating static images of these visualizations the Selenium package is required (v.4.28.0, https://www.selenium.dev/documentation/)

```
pip install selenium
```

## This exampleCities are used as the basis to find random samples of intersections. The region and country names are nice to have, but they are not necessary.

In [1]:
# region <set up parameters>

import os
import csv
import pandas as pd
import multiprocessing

sample_size = 1
min_distance = 2
random_seed = 4
network_type = 'drive'
point_distance_size = 10000
experiment_name = "2025-07-evaluation"
base_path=f"/home/arvidh/Documents/GitHub/proj_full-analysis/evaluation/{experiment_name}"
print(base_path)
min_od_distance = 4750
max_od_distance = 5250
od_pair_sample_size = 3


if not os.path.exists(base_path):
    os.makedirs(base_path)

parameters_file_path = os.path.join(base_path, f"parameters.csv")
city_sample_nodes_path = os.path.join(base_path, f'city_sample_nodes_{experiment_name}.csv')
local_graph_folder = os.path.join(base_path, 'local_origin_graphs')

with open(parameters_file_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(["Parameter", "Value"])
    writer.writerow(["sample_size", sample_size])
    writer.writerow(["min_distance", min_distance])
    writer.writerow(["random_seed", random_seed])
    writer.writerow(["network_type", network_type])
    writer.writerow(["point_distance_size", point_distance_size])
    writer.writerow(["min_od_distance", min_od_distance])
    writer.writerow(["max_od_distance", max_od_distance])
    writer.writerow(["base_path", base_path])
    writer.writerow(["city_sample_nodes_path", city_sample_nodes_path])
    writer.writerow(["local_graph_folder", local_graph_folder])
    writer.writerow(["base_path", base_path])
    writer.writerow(["experiment_name", experiment_name])


param = pd.read_csv(parameters_file_path)
display(param)



num_processes = multiprocessing.cpu_count()  # Adjust based on your system's capabilities
print(f"Number of processes to use: {num_processes}")

# endregion

/home/arvidh/Documents/GitHub/proj_full-analysis/evaluation/2025-07-evaluation


,Parameter,Value
0,sample_size,1
1,min_distance,2
2,random_seed,4
3,network_type,drive
4,point_distance_size,10000
5,min_od_distance,4750
6,max_od_distance,5250
7,base_path,/home/arvidh/Documents/GitHub/proj_full-analys...
8,city_sample_nodes_path,/home/arvidh/Documents/GitHub/proj_full-analys...
9,local_graph_folder,/home/arvidh/Documents/GitHub/proj_full-analys...


Number of processes to use: 8


In [2]:
# region <get node sample from cities>
# The workflow for analyzing the routes begins with coordinate points used as origin locations.
import os 
import pandas as pd
from route_network_analysis import node_sampling
df = pd.read_csv("8_city_sample.csv")

display(df)

city_sample_nodes_path = os.path.join(base_path, f'city_sample_nodes_{experiment_name}.csv')

if os.path.exists(city_sample_nodes_path):
    print("sample nodes already retrieved")
else:
    node_sample_df = node_sampling.get_random_nodes_for_all_cities(df, min_distance_km=min_distance, sample_size=sample_size,random_seed=random_seed)
    node_sample_df['graph_path'] = node_sample_df.apply(lambda row: os.path.join(local_graph_folder, f"{row['city_name_en']}_{row['node_id']}.graphml"), axis=1)
    node_sample_df.to_csv(os.path.join(base_path, city_sample_nodes_path))

# endregion

,city_name_en,country_name_en,region,city_name,country_name,continent,latitude,longitude
0,Buenos Aires,Argentina,Latin America,Buenos Aires,Argentina,South America,-33.365800,-60.199550
1,Miami,United States,US/Canada,Miami,United States,North America,25.774173,-80.193620
2,Rome,Italy,Europe,Roma,Italia,Europe,41.893320,12.482932
3,Barcelona,Spain,Europe,Barcelona,España,Europe,41.382894,2.177432
4,Tehran,Iran,Middle East/Africa,شهر تهران,ایران,Asia,35.687720,51.439639
5,Istanbul,Turkey,Middle East/Africa,İstanbul,Türkiye,Asia,41.006381,28.975872
6,Melbourne,Australia,Asia/Oceania,Melbourne,Australia,Oceania,-37.814245,144.963173
7,Shanghai,China,Asia/Oceania,Shanghai,中国,Asia,31.231271,121.470015


Random nodes for Buenos Aires that are at least 2 km apart: [1406097496]
Random nodes for Buenos Aires: [1406097496]
Node: 1406097496
Random node coordinates for Buenos Aires: -33.3550713, -60.2680864
Random nodes for Miami that are at least 2 km apart: [99231070]
Random nodes for Miami: [99231070]
Node: 99231070
Random node coordinates for Miami: 25.8029559, -80.1854068
Random nodes for Rome that are at least 2 km apart: [298569040]
Random nodes for Rome: [298569040]
Node: 298569040
Random node coordinates for Rome: 41.8864835, 12.5405169
Random nodes for Barcelona that are at least 2 km apart: [598225120]
Random nodes for Barcelona: [598225120]
Node: 598225120
Random node coordinates for Barcelona: 41.4361111, 2.1886746
Random nodes for Tehran that are at least 2 km apart: [2090045873]
Random nodes for Tehran: [2090045873]
Node: 2090045873
Random node coordinates for Tehran: 35.6619292, 51.4375593
Random nodes for Istanbul that are at least 2 km apart: [1057860799]
Random nodes for I

In [3]:
# region <create graphs>
import os  # for file operations
import pandas as pd  # for reading the csv file
import joblib
import logging
import ast
import route_network_analysis as rna
import osmnx as ox
logging.basicConfig(level=logging.ERROR, format='%(asctime)s - %(levelname)s - %(message)s',filename='jupyter.log', filemode='w')

#
sub_folder = "local_origin_graphs"
local_graph_folder = os.path.join(base_path, sub_folder)
if not os.path.exists(local_graph_folder):
    os.makedirs(local_graph_folder)


df = pd.read_csv(city_sample_nodes_path)

def create_graphs(row):
    if os.path.exists(row['graph_path']):
        print(f"Graph exists: {row['graph_path']}")
        return True, row['city_name'], row['node_id']
    else:
        print(f"---Graph missing: {row['graph_path']}---")
        # Apply the function asynchronously
    try:
        latlon_point = ast.literal_eval(row['node_latlon'])
        og = rna.origin_graph(origin_point=latlon_point, distance_from_point=point_distance_size,
                          city_name=row["city_name_en"], network_type=network_type, remove_parallel=True, simplify=True)
    
        og.save_graph(row['graph_path'])
    
        # Plot the origin graph to see if something is obviously wrong
        ox.plot_graph(og.graph, node_color='blue', node_size=5, edge_linewidth=1, edge_color='black', bgcolor='white',
                       save=True, filepath=os.path.join(local_graph_folder, f"{row['city_name']}_{row['node_id']}.png"), show=False)
        logging.error(f"Finished with graph: {row['graph_path']}")
        return True, row['city_name'], row['node_id']

    except Exception as e:
        logging.error(f"error {e} creating {row['graph_path']}")
        return False, row['city_name'], row['node_id'], e
        

# Number of processes to use
num_processes = (joblib.cpu_count()-2)
print(f"Number of processes to use: {num_processes}")

# Collect results from joblib
results = []
try:

    results = joblib.Parallel(n_jobs=num_processes,backend='loky')(
        joblib.delayed(create_graphs)(row) for _, row in df.iterrows()
    )
except Exception as e:
    print(f"Joblib parallel processing error: {e}")
    
for result in results:
    if not result[0]: 
        print(f"failed creating graph {result[1]}{result[2]}. error {result[3]}")
    else:
        print(f"finished creating graph {result[1]}{result[2]}")

print("finished")

# endregion

Number of processes to use: 6
---Graph missing: /home/arvidh/Documents/GitHub/proj_full-analysis/evaluation/2025-07-evaluation/local_origin_graphs/Buenos Aires_1406097496.graphml---
---Graph missing: /home/arvidh/Documents/GitHub/proj_full-analysis/evaluation/2025-07-evaluation/local_origin_graphs/Melbourne_319642457.graphml---
finished creating graph Buenos Aires1406097496
finished creating graph Miami99231070
finished creating graph Rome298569040
finished creating graph Barcelona598225120
finished creating graph Tehran2090045873
finished creating graph Istanbul1057860799
finished creating graph Melbourne319642457
finished creating graph Shanghai622623813
finished


In [4]:
# The next step is to add weights to the edges of the graph.
import pandas as pd # for reading the csv file
import joblib # replacing multiprocessing with joblib
df = pd.read_csv(city_sample_nodes_path)
df['weights_added'] = False

import route_network_analysis as rna

#if 'weights_added' not in df.columns:
#    df['weights_added'] = False
def add_graph_weights(row):
    og = rna.origin_graph.from_graphml(graphml_path=row['graph_path'])
    og.add_simplest_paths_from_origin()
    og.add_weights('deviation_from_prototypical')
    og.add_weights('node_degree')
    og.add_weights('instruction_equivalent')
    og.add_weights('betweenness_centrality')
    og.save_graph(row['graph_path'])
    print(f"Finished with graph: {row['city_name_en']} node: {row['node_id']}",flush=True)
    return True, row['city_name_en'], row['node_id']

# Number of processes to use
num_processes = (joblib.cpu_count()-2)
print(f"Number of processes to use: {num_processes}")
# Collect results from joblib
rows_to_process = []
for idx, row in df.iterrows():
    rows_to_process.append(row)


results = joblib.Parallel(n_jobs=num_processes, backend='loky')(
    joblib.delayed(add_graph_weights)(row) for row in rows_to_process
)


for result in results:
    if result[0]:
        mask = (df['city_name_en'] == result[1]) & (df['node_id'] == result[2])
        df.loc[mask, 'weights_added'] = True
df.to_csv(city_sample_nodes_path)

Number of processes to use: 6
---Graph missing: /home/arvidh/Documents/GitHub/proj_full-analysis/evaluation/2025-07-evaluation/local_origin_graphs/Barcelona_598225120.graphml---
Finished with graph: Buenos Aires node: 1406097496
---Graph missing: /home/arvidh/Documents/GitHub/proj_full-analysis/evaluation/2025-07-evaluation/local_origin_graphs/Miami_99231070.graphml---
---Graph missing: /home/arvidh/Documents/GitHub/proj_full-analysis/evaluation/2025-07-evaluation/local_origin_graphs/Shanghai_622623813.graphml---
Finished with graph: Miami node: 99231070


KeyboardInterrupt: 

In [1]:
import pandas as pd # for reading the csv file
import joblib # replacing multiprocessing with joblib
import route_network_analysis as rna

df = pd.read_csv(city_sample_nodes_path)
display(df)
local_odpair_folder = os.path.join(base_path, "od_pair_data")
local_odpair_base_folder = os.path.join(local_odpair_folder, 'base')
local_odpair_geom_folder = os.path.join(local_odpair_folder, 'geom')
print(f"odpair data will be stored at {local_odpair_folder}")
os.makedirs(local_odpair_folder, exist_ok=True)
os.makedirs(local_odpair_base_folder, exist_ok=True)
os.makedirs(local_odpair_geom_folder, exist_ok=True)


df['od_pairs_added'] = False


def get_od_pairs(row):
    print(f"Finding OD_pairs for graph: {row['city_name_en']} node: {row['node_id']}",flush=True)
    og = rna.origin_graph.from_graphml(graphml_path=row['graph_path'])
    og.create_od_pairs(min_radius=min_od_distance, max_radius=max_od_distance, sample_size=od_pair_sample_size)
    od_pair_data = og.get_od_pair_data()
    od_pair_geom_data = og.get_od_pair_geom_data()
    json_base_path = os.path.join(local_odpair_base_folder, f"od_pair_{row['city_name_en']}_{row['node_id']}.json")
    json_geom_path = os.path.join(local_odpair_geom_folder, f"od_pair_geom_{row['city_name_en']}_{row['node_id']}.json")
    od_pair_data.to_json(json_base_path, orient="records", default_handler=str, indent=2)
    od_pair_geom_data.to_json(json_geom_path, orient="records", default_handler=str, indent=2)
    print(f"Finished finding OD_pairs for graph: {row['city_name_en']} node: {row['node_id']}",flush=True)
    return True,row['city_name_en'],row['node_id']


num_processes = (joblib.cpu_count() - 4)
print(f"Number of processes to use: {num_processes}")



rows_to_process = []
for idx, row in df.iterrows():
    #if not row['od_pairs_added']:
        rows_to_process.append(row)

results = joblib.Parallel(n_jobs=num_processes, backend='loky')(
    joblib.delayed(get_od_pairs)(row) for row in rows_to_process
)

for result in results:
    if result[0]:
        mask = (df['city_name_en'] == result[1]) & (df['node_id'] == result[2])
        df.loc[mask, 'od_pairs_added'] = True


import glob


# Get all json files from od_pair_data folder
od_pair_base_files = glob.glob(os.path.join(local_odpair_base_folder, "*.json"))
od_pair_geom_files = glob.glob(os.path.join(local_odpair_geom_folder, "*.json"))
# Read and combine all json files
od_pair_base_data = pd.concat([pd.read_json(f) for f in od_pair_base_files], ignore_index=False)
od_pair_geom_data = pd.concat([pd.read_json(f) for f in od_pair_geom_files], ignore_index=False)

print(f"Total number of od-pairs: {len(od_pair_base_data)}")
print(od_pair_base_data.columns)


# The od-pair data contains lists and dictionaries that are not easily saved to a csv file, so we store it as a json file.
# Still, there some columns that need to be serialized to strings such as shapely polygon objects.



od_pair_data_geom_path_json = os.path.join(local_odpair_folder, 'origin_od_pair_geom.json')
od_pair_data_base_path_csv = os.path.join(local_odpair_folder, 'origin_od_pair_base.csv')

od_pair_base_data.to_csv(od_pair_data_base_path_csv)
od_pair_geom_data.to_json(od_pair_data_geom_path_json, orient="records", default_handler=str, indent=2)

print("done")



NameError: name 'city_sample_nodes_path' is not defined